# Test-split evaluation of the v13 runs

Scores the v13 runs (baseline YOLOv11n 640, YOLOv11s 1024, YOLOv11s 1024 + TTA; seeds 0-2) on
the full v13 test split, with Welch tests and Holm correction, and measures the density law on
the v13 validation split. This is the v13 counterpart of `eval_test.py` + `eval_test_seeds.py` +
`build_reviewer_stats.py` (R1-2).

Prerequisite: the six core v13 notebooks have finished (`nb/v13_baseline_s{0,1,2}_640_W2.ipynb`,
`nb/v13_small_s{0,1,2}_1024_W2.ipynb`).

**When finished,** restart the kernel. Ultralytics' `val()` can leave dataloader workers alive.


In [ ]:
# === Environment check ===
import os, sys, gc, csv, json, time
from pathlib import Path
import torch, psutil
# Plain tqdm, NOT tqdm.auto: inside a notebook kernel tqdm.auto resolves to the ipywidgets
# frontend, and VS Code then blocks on a "Widgets require us to download supporting files from a
# 3rd party website" dialog (@jupyter-widgets/base). The text bar needs no CDN and no network.
from tqdm import tqdm

print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "no CUDA - qml venv needs torch cu130"
print("gpu        :", torch.cuda.get_device_name(0),
      "| cc", torch.cuda.get_device_capability(0),
      "| %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
vm = psutil.virtual_memory()
print("system RAM : %.1f GB total, %.1f GB available (%.0f%% used)"
      % (vm.total/1e9, vm.available/1e9, vm.percent))


In [ ]:
# === Config ===
import math
from collections import Counter, defaultdict
import numpy as np, pandas as pd, yaml
from scipy import stats

HERE    = Path(r"F:\0BScDefense\04_Code\journal_analysis\v12_local")
DATASET = "v13"
DATA    = Path(r"f:\0BScDefense\05_Datasets\merged-dataset-v13")          # dataset root, holds data.yaml
STEM    = "eval_v13"                 # results/<STEM>_*.json|csv|md
os.chdir(HERE)
sys.path.insert(0, str(HERE))
from eval_ensemble import NAMES, GROUP_B
from build_reviewer_stats import welch_ci

WORKERS  = 0      # dataloader workers per val(); 0 = none left behind in this long-lived kernel
MIN_INST = 5      # a class enters the density correlation only with >= 5 boxes in the subset
SEEDS    = (0, 1, 2)
SYSTEMS  = {      # system -> (run-name pattern, imgsz, TTA)
    "baseline":   ("v13_baseline_s{seed}_640_W2", 640, False),
    "s@1024":     ("v13_small_s{seed}_1024_W2", 1024, False),
    "s@1024+TTA": ("v13_small_s{seed}_1024_W2", 1024, True),
}
KEYS = ("ALL", "A", "B", "recall", "mAP50_95")
RES = HERE / "results"
SPL = HERE / "splits"; SPL.mkdir(exist_ok=True)


def weights(system, seed):
    return HERE / "runs" / SYSTEMS[system][0].format(seed=seed) / "weights" / "best.pt"


missing = [str(weights(s, k)) for s in SYSTEMS for k in SEEDS if not weights(s, k).exists()]
assert not missing, "missing weights:\n" + "\n".join(missing)
assert (DATA / "data.yaml").exists(), DATA
print(f"dataset {DATASET} | {len(SYSTEMS)} systems x {len(SEEDS)} seeds | weights found")


In [ ]:
# === Evaluation subsets ===
# name -> (data yaml, split argument for val(), image paths)
def images(split):
    return sorted(p for p in (DATA / "images" / split).iterdir()
                  if p.suffix.lower() in (".jpg", ".jpeg", ".png"))


SUBSETS = {"test_full": (DATA / "data.yaml", "test", images("test")),
           "valid_full": (DATA / "data.yaml", "val", images("valid"))}

if DATASET == "v12":
    pairs = pd.read_csv(RES / "dihedral_twins_pairs.csv")

    def with_train_twin(confirmed_only):
        """(split, file) of every valid/test image whose twin group reaches into train."""
        parent = {}

        def find(x):
            parent.setdefault(x, x)
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        P = pairs[pairs.confirmed == 1] if confirmed_only else pairs
        for r in P.itertuples():
            parent[find((r.split_a, r.img_a))] = find((r.split_b, r.img_b))
        roots = {find(n) for n in list(parent) if n[0] == "train"}
        return {n for n in list(parent) if n[0] != "train" and find(n) in roots}

    lenient, strict = with_train_twin(False), with_train_twin(True)
    for split, arg in (("valid", "val"), ("test", "test")):
        imgs = images(split)
        for tag, keep in (("twinfree", lambda p: (split, p.name) not in lenient),
                          ("twinned", lambda p: (split, p.name) in strict)):
            name = f"{split}_{tag}"
            lst = [p for p in imgs if keep(p)]
            txt = SPL / f"v12_{name}.txt"
            txt.write_text("\n".join(str(p) for p in lst) + "\n", encoding="utf-8")
            yml = SPL / f"v12_{name}.yaml"
            yml.write_text(yaml.safe_dump(
                {"path": str(DATA), "train": "images/train", "val": str(txt), "test": str(txt),
                 "nc": len(NAMES), "names": NAMES}, sort_keys=False), encoding="utf-8")
            SUBSETS[name] = (yml, "test", lst)

# boxes per class in each subset, to flag classes too thin to score
INST = {}
for name, (_, _, lst) in SUBSETS.items():
    c = Counter()
    for p in lst:
        lbl = p.parent.parent.parent / "labels" / p.parent.name / (p.stem + ".txt")
        if lbl.exists():
            c.update(NAMES[int(l.split()[0])] for l in lbl.read_text().splitlines() if l.strip())
    INST[name] = dict(c)
inst = pd.DataFrame(INST).fillna(0).astype(int).reindex(NAMES)
inst.loc["images"] = [len(SUBSETS[n][2]) for n in inst.columns]
inst.to_csv(RES / f"{STEM}_subset_boxes.csv")
display(inst)


In [ ]:
# === Evaluate (resumable) ===
from ultralytics import YOLO

JOBS = [(sub, system, seed) for sub in SUBSETS for system in SYSTEMS for seed in SEEDS
        if sub.startswith("test") or system == "baseline"]
out_json = RES / f"{STEM}_eval.json"
done = json.loads(out_json.read_text()) if out_json.exists() else []
have = {(r["subset"], r["system"], r["seed"]) for r in done}
todo = [j for j in JOBS if j not in have]
print(f"{len(JOBS)} jobs, {len(todo)} still to run")

bar = tqdm(todo, unit="job", file=sys.stdout, dynamic_ncols=True)
for sub, system, seed in bar:
    bar.set_description(f"{sub} | {system} s{seed}")
    _, imgsz, tta = SYSTEMS[system]
    yml, split, lst = SUBSETS[sub]
    b = YOLO(str(weights(system, seed))).val(
        data=str(yml), split=split, imgsz=imgsz, augment=tta, workers=WORKERS, plots=False,
        verbose=False, project=str(HERE / "runs"), name="eval_tmp", exist_ok=True).box
    ap50 = {NAMES[int(c)]: float(b.ap50[i]) for i, c in enumerate(b.ap_class_index)}
    ap95 = {NAMES[int(c)]: float(b.ap[i]) for i, c in enumerate(b.ap_class_index)}
    ga = [v for k, v in ap50.items() if k not in GROUP_B]
    gb = [v for k, v in ap50.items() if k in GROUP_B]
    done.append({"subset": sub, "system": system, "seed": seed, "n_images": len(lst),
                 "ALL": float(b.map50), "A": float(np.mean(ga)) if ga else math.nan,
                 "B": float(np.mean(gb)) if gb else math.nan, "recall": float(b.mr),
                 "mAP50_95": float(b.map), "n_classes": len(ap50), "ap50": ap50, "ap50_95": ap95})
    out_json.write_text(json.dumps(done, indent=1))      # unrounded; saved after every job
    bar.set_postfix({"mAP50": f"{float(b.map50):.4f}",
                     "RAM%": f"{psutil.virtual_memory().percent:.0f}"})
    gc.collect(); torch.cuda.empty_cache()
bar.close()
R = pd.DataFrame(done)
display(R.groupby(["subset", "system"])[list(KEYS)].mean().round(4))


In [ ]:
# === Welch tests: each system vs the baseline, per test subset ===
def holm(ps):
    adj, run = {}, 0.0
    for i, k in enumerate(sorted(ps, key=ps.get)):
        run = max(run, min(1.0, (len(ps) - i) * ps[k]))
        adj[k] = run
    return adj


STATS = {}
for sub in [s for s in SUBSETS if s.startswith("test")]:
    T = R[R.subset == sub]
    base = T[T.system == "baseline"].sort_values("seed")
    STATS[sub] = {}
    print(f"\n=== {sub}: {len(SUBSETS[sub][2])} images ===")
    for system in [s for s in SYSTEMS if s != "baseline"]:
        arm = T[T.system == system].sort_values("seed")
        rows, ps = {}, {}
        for k in KEYS:
            d, lo, hi, dfree, _ = welch_ci(base[k].values, arm[k].values)
            p = float(stats.ttest_ind(arm[k].values, base[k].values, equal_var=False).pvalue)
            ps[k] = p
            rows[k] = {"baseline": float(base[k].mean()), "baseline_sd": float(base[k].std(ddof=1)),
                       "system": float(arm[k].mean()), "system_sd": float(arm[k].std(ddof=1)),
                       "delta": d, "ci95": [lo, hi], "welch_df": dfree, "p": p}
        for k, a in holm(ps).items():
            rows[k]["p_holm"] = a
        STATS[sub][system] = rows
        print(f"  {system}")
        for k in KEYS:
            r = rows[k]
            print(f"    {k:9s} {r['baseline']:.3f} -> {r['system']:.3f}   {r['delta']:+.3f} "
                  f"[{r['ci95'][0]:+.3f}, {r['ci95'][1]:+.3f}]   p {r['p']:.3f}   Holm {r['p_holm']:.3f}")
_ = (RES / f"{STEM}_stats.json").write_text(json.dumps(STATS, indent=1))


In [ ]:
# === Density law: per-class mAP@50 of the baseline vs boxes per image ===
# boxes per image = mean number of a class's boxes over the images containing it, all splits
# pooled (the definition of analyse_protocol.py, recomputed here from this dataset's labels).
per = defaultdict(list)
for split in ("train", "valid", "test"):
    for lbl in (DATA / "labels" / split).glob("*.txt"):
        c = Counter(int(l.split()[0]) for l in lbl.read_text().splitlines() if l.strip())
        for k, n in c.items():
            per[NAMES[k]].append(n)
DENS = {k: float(np.mean(v)) for k, v in per.items()}

rows = []
for sub in [s for s in ("valid_full", "valid_twinfree", "test_full", "test_twinfree") if s in SUBSETS]:
    B = R[(R.subset == sub) & (R.system == "baseline")]
    ap = pd.DataFrame(list(B.ap50)).mean()
    cls = [c for c in ap.index if INST[sub].get(c, 0) >= MIN_INST]
    x = np.array([DENS[c] for c in cls]); y = ap[cls].values
    r, p = stats.pearsonr(x, y)
    rho, prho = stats.spearmanr(x, y)
    rows.append({"subset": sub, "classes": len(cls), "pearson_r": r, "p": p,
                 "spearman_rho": rho, "p_spearman": prho,
                 "dropped_thin_classes": [c for c in ap.index if c not in cls]})
    print(f"{sub:15s} {len(cls):2d} classes   r = {r:+.3f} (p = {p:.1e})   "
          f"Spearman {rho:+.3f} (p = {prho:.1e})")
DENSITY = pd.DataFrame(rows)
DENSITY.to_csv(RES / f"{STEM}_density.csv", index=False)


In [ ]:
# === Verdict ===
def claim(sub, system, k):
    r = STATS[sub][system][k]
    lo, hi = r["ci95"]
    ok = r["p"] < 0.05 and (lo > 0 or hi < 0)
    return (f"{system} vs baseline, {k}: {r['delta']:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}], "
            f"p = {r['p']:.3f}, Holm p = {r['p_holm']:.3f} -> "
            + ("significant" + ("" if r["p_holm"] < 0.05 else ", but not after Holm")
               if ok else "NOT significant"))

lines = [f"# {STEM}: summary", ""]
main = "test_twinfree" if "test_twinfree" in STATS else "test_full"
for sub in [s for s in ("test_full", "test_twinfree", "test_twinned") if s in STATS]:
    lines += [f"## {sub} ({len(SUBSETS[sub][2])} images)", ""]
    lines += [f"- {claim(sub, 's@1024+TTA', k)}" for k in ("mAP50_95", "B", "ALL")]
    lines += [f"- {claim(sub, 's@1024', k)}" for k in ("mAP50_95", "B")]
    lines.append("")
lines += ["## Density law (baseline, per-class mAP@50 vs boxes per image)", ""]
lines += [f"- {r.subset}: r = {r.pearson_r:+.3f} (p = {r.p:.1e}), {r.classes} classes"
          for r in DENSITY.itertuples()]
if DATASET == "v12":
    lines += ["", "## Leakage effect (test, same class, twinned - twin-free AP@50)", ""]
    lines += [f"- {s}: {v['per_class_mean']:+.3f}" for s, v in LEAK.items()]
text = "\n".join(lines) + "\n"
(RES / f"{STEM}_summary.md").write_text(text, encoding="utf-8")
print(text)
print(f"headline subset for the paper: {main}")


## Next

Take the summary above to `../TWIN_LEAKAGE_PLAN.md`, decision gate after step 1 (v12) or step 4
(v13). Then restart the kernel.
